# Definitions

In [121]:
### Import Modules
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Gate, Instruction
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Statevector
from qiskit.visualization import circuit_drawer
import qiskit.qpy as qpy
import numpy as np
import math
# import sys
# print(sys.platform)

## Parameters

In [122]:
### Parameters (4,2,16) (3,1,8) (2,1,4)
n = 2 # Number of qubits for |j⟩
# Having 1 neighbor as |k⟩=|0⟩, and having 2 neighbors as |k⟩=|1⟩
m = 1  # Number of qubits for |k⟩ (m << n)
# Precision level (number of qubits) for representing x_v
p = n

In [123]:
### Network characteristics
N = 2**n  # Number of nodes
s = 2**m  # Sparsity level
P = 2**p  # X scaling factor

### Adjacency matrix
A = np.zeros((N, N), dtype=int)

## Matrices Options

In [124]:
### Matrix option 1: random with less or equal to s neighbors

# np.random.seed(42)  # For reproducibility
# for i in range(N):
#     adjacency_list = np.random.choice(np.delete(np.arange(N), i), size=min(s, N-1), replace=False)
#     A[i, adjacency_list] = 1
# A[-1, :] = 0  # Force the last node to have no neighbors
# A = np.minimum(A, A.T)  # Ensure symmetry

### E structure (like an edge list)
# E_list = [np.nonzero(row)[0].tolist() for row in A]
# adjacency_list = [E_list[v] + [N-1] * (s - len(E_list[v])) for v in range(N)] # Fill missing neighbors with N-1 (last node)
# print("\nE_list: ", E_list)
# print("\nadjacency_list: ", adjacency_list)

In [125]:
#### Matrix Option 2: Circulant graph (subtype of Cayley graph)
# defined by a set of shift operations that dictate node connection in a cyclic manner 

adjacency_list = []
shifts = [1, -1]  # Node is connect to the previous and next one
if s > 2: # For s=4, shifts = [1, 2, -1, -2]
    shifts = list(range(1, s//2 + 1)) + [-i for i in range(1, s//2 + 1)]
for i in range(N):
    v_list = []
    for shift in shifts:
        j = (i + shift) % N  # Wrap around (cycle structure)
        v_list.append(j)
        A[i, j] = 1
        A[j, i] = 1 
    adjacency_list.append(v_list) 

In [126]:
print("\nAdjacency Matrix (Sparse): \n", A)
print("\nadjacency_list: \n", adjacency_list)

### Nodes degrees
nodes_degrees = [len(e) for e in adjacency_list]
print("\nnode_degrees: ", nodes_degrees)
# Chosen for simplicity: if all nodes have the same degree, we shall divide by degree squared.
nodes_degrees_c1 = [k**2 for k in nodes_degrees]

### Feature vector
# Be careful: if the features are not these, issues may arise
# In particular, if features[N] is not 0, oracle O_L shall be adjusted
features = np.round(np.linspace(1 - 1/P, 0, N), 10)
features_int = (features * P).astype(int)
print("\nFeature Vector: \n", features)
print("\nFeature Vector (scaled by P): \n", features_int)


Adjacency Matrix (Sparse): 
 [[0 1 0 1]
 [1 0 1 0]
 [0 1 0 1]
 [1 0 1 0]]

adjacency_list: 
 [[1, 3], [2, 0], [3, 1], [0, 2]]

node_degrees:  [2, 2, 2, 2]

Feature Vector: 
 [0.75 0.5  0.25 0.  ]

Feature Vector (scaled by P): 
 [3 2 1 0]


## Binary Conversion

In [127]:
def f_x_binary(float):
    ''' Multiply by the scaling factor (should be integer, and int forces that)
        Convert to p-bit binary representation
        Slicing [2:] removes the '0b' prefix
        zfill(p) ensures exactly p bits (pad with 0s if needed).'''
    return bin(int(float * P))[2:].zfill(p) 

def check_f_x_binary(vec, P):
    ''' Check if the functions f_x_binary and f_x_binary_inverse are working properly.'''
    vec_bin = [f_x_binary(x) for x in vec]
    vec_bin_inv = [(int(x_bin, 2) / P) for x_bin in vec_bin]
    # Check if all elements match
    all_match = all(np.isclose(vec[i], vec_bin_inv[i]) for i in range(len(vec)))
    print("\nDo all elements (between vec and vec_bin_inv) match? ", "Yes" if all_match else "No")
    if not all_match:
        for i in range(len(vec)):
            if not np.isclose(vec[i], vec_bin_inv[i]):
                print(f"Mismatch at index {i}: Original = {vec[i]}, Decoded = {vec_bin_inv[i]}")

# Simulation

## Access Statevector

In [128]:
def access_statevector(qc, flag_c=True, flag_i=True, flag_a=False, flag_l=True):
    ''' Generates the quantum statevector resulting from executing the quantum circuit qc.
        If flag=True, shows all substates, including non-zero qubits. '''
    
    statevector = Statevector.from_instruction(qc)

    print("\nExtracted Information from Useful States:")
    for idx, amplitude in enumerate(statevector):
        if amplitude.real == 0.00:  
            continue  # Hide states

        state_bin = format(idx, f'0{qc.num_qubits}b')

        if (flag_c or state_bin[n + 2]) and \
           (flag_i or (state_bin[n] == '0' and state_bin[n + 1] == '0')) and \
           (flag_l or all(state_bin[n + 3 + i] == '0' for i in range(2*m))) and \
           (flag_a or all(state_bin[-i] == '0' for i in range(1, 6))):

            v = int(state_bin[:n], 2)
            c = int(state_bin[n + 2:n + 3], 2)
            l1 = int(state_bin[n + 3:n + 3 + m], 2)
            l2 = int(state_bin[n + 3 + m:n + 3 + 2*m], 2)
            u1 = int(state_bin[n + 3 + 2*m:2*n + 3 + 2*m], 2)
            u2 = int(state_bin[2*n + 3 + 2*m:3*n + 3 + 2*m], 2)
            k_v = int(state_bin[3*n + 3 + 2*m:3*n + 3 + 4*m], 2)
            x_u = int(state_bin[3*n + 3 + 4*m:3*n + 3 + 4*m + p], 2) / P

            print(f"State: |{state_bin}>, Amplitude: {round(amplitude.real, 10)}, "
                  f"v: {v}, c: {c}, l1: {l1}, l2: {l2}, u1: {u1}, u2: {u2}, x_u: {x_u}, k_v: {k_v}")

    '''

    qr_v = QuantumRegister(n, 'v')  # Basis-encode node index
    qr_i = QuantumRegister(2, 'i')  # Basis-encode power (i) of feature value (x^i)
    qr_c = QuantumRegister(1, 'c')  # Basis-encode power (c) of matrix A (A^c)
    qr_l1 = QuantumRegister(m, 'l1')  # Basis-encode non-zero entries of each row of A
    qr_l2 = QuantumRegister(m, 'l2')  # Basis-encode non-zero entries of each row of A^2
    qr_u1 = QuantumRegister(n, 'u1')  # Basis-encode node's l-neighbor index of v (auxiliary)
    qr_u2 = QuantumRegister(n, 'u2')  # Basis-encode node's l-neighbor index of u (auxiliary)
    qr_k = QuantumRegister(2*m, 'k')  # Basis-encode degree value (k) [!]
    qr_z = QuantumRegister(p, 'z')  # Basis-encode feature value (x)
    qr_a = QuantumRegister(5, 'a')  # Ancilla qubit for rotation
    
    qc = QuantumCircuit(qr_a, qr_z, qr_k, qr_u2, qr_u1, qr_l2, qr_l1, qr_c, qr_i, qr_v)

    state_bin: [qr_v][qr_i]qr_c[qr_l1][qr_l2][qr_u1][qr_u2][qr_k][qr_z][qr_a] (n,m,N)
    state_bin: 1111   01    0    10     01    0001   0011   11    0110  00000 (4,2,16)
    state_bin: 111    01    0    1      0     001    011    1     010   00000 (3,1,8)

    '''

## Integer to Register Gate

In [129]:
class IntegerToRegisterGate(Gate):
    ''' 
        Gate to take an integer and encode it into a quantum register.
        Set a register to an integer value via X gates.
    '''
    def __init__(self, name, integer, num_qubits): # Initialize gate
        max_value = 2**num_qubits - 1
        if integer < 0 or integer > max_value:
            raise ValueError(f"Integer {integer} cannot be represented with {num_qubits} qubits")

        self.integer = integer
        self.num_qubits = num_qubits
        super().__init__(name, num_qubits, [integer]) # 0 classical bits, 1 param (the integer)
        self._define()

    def _define(self): # Behaviour/operations of the gate
        binary = format(self.integer, f'0{self.num_qubits}b')
        qc = QuantumCircuit(self.num_qubits, name=self.name)
        [qc.x(i) for i, bit in enumerate(reversed(binary)) if bit == '1']
        self.definition = qc  # Quantum Circuit

    def inverse(self):
        return self  # X gates are self-inverse

## Feature Rotation Gate

In [130]:
def FeatureRotation(thetas, control_size, gate_name):
    ''' Controlled rotation of one qubit based on a register with nr_qubits.
        Here, control_size = 2**nr_qubits.
    '''
    print("\nThetas (F_X): ", thetas)
    unitary = np.eye(2 * control_size, dtype=complex)  

    for x, theta in enumerate(thetas):
        W = np.array([
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2), np.cos(theta / 2)]
        ])
        id = 2 * x  # Pair of rows/cols for each x
        unitary[id:(id + 2), id:(id + 2)] = W

    return UnitaryGate(unitary, label=gate_name)

## Oracles (Basis-Encoders)

In [131]:
########################################## Oracle O_L ###############################################

def oracle_O_L(adjacency_list, n, m):
    ''' Return an oracle O_L Gate that encodes r(v,l) into qr_u, controlled on qr_v and qr_l. '''
    qc = QuantumCircuit(m + n + n, name='O_L')  # m qubits (l), n qubits (v), n qubits (u output)

    for v in range(len(adjacency_list)):
        for l, u in enumerate(adjacency_list[v]):
            subgate = IntegerToRegisterGate('O_L_val', u, n)
            ctrl_gate = subgate.control(num_ctrl_qubits=m + n, ctrl_state=f'{(l + v * (2 ** m)):0{m + n}b}')
            qc.append(ctrl_gate, list(range(m + n + n)))
            
    return qc.to_gate(label='O_L')

gate_O_L = oracle_O_L(adjacency_list, n, m)
print(gate_O_L.definition)

                                                                 »
q_0: ───────o──────────────■──────────────o──────────────■───────»
            │              │              │              │       »
q_1: ───────o──────────────o──────────────■──────────────■───────»
            │              │              │              │       »
q_2: ───────o──────────────o──────────────o──────────────o───────»
     ┌──────┴──────┐┌──────┴──────┐┌──────┴──────┐┌──────┴──────┐»
q_3: ┤0            ├┤0            ├┤0            ├┤0            ├»
     │  O_l_val(1) ││  O_l_val(3) ││  O_l_val(2) ││  O_l_val(0) │»
q_4: ┤1            ├┤1            ├┤1            ├┤1            ├»
     └─────────────┘└─────────────┘└─────────────┘└─────────────┘»
«                                                                 
«q_0: ───────o──────────────■──────────────o──────────────■───────
«            │              │              │              │       
«q_1: ───────o──────────────o──────────────■──────────────■───

In [132]:
########################################## Oracle O_X ###############################################

def oracle_O_X(features, n, p, P, ctrl_val):
    '''Return an oracle_O_X gate that encodes feature x_u into qr_z, controlled on qr_c and qr_u: '''
    qc = QuantumCircuit(1 + n + p, name=f'O_X')

    for u, x_u in enumerate(features):
        subgate = IntegerToRegisterGate(f'x_u', int(x_u * P), p)

        # control states: qr_c=0 for u1, qr_c=1 for u2
        ctrl_gate = subgate.control(num_ctrl_qubits = 1 + n, ctrl_state = f'{(ctrl_val + u * 2):0{1 + n}b}')
        qc.append(ctrl_gate, list(range(1 + n + p)))

    return qc.to_gate(label=f'O_X')

ctrl_gate_O_X_c0 = oracle_O_X(features, n, p, P, ctrl_val=0) # If ctrl_val == 0: encodes to qr_z from qr_u1
ctrl_gate_O_X_c1 = oracle_O_X(features, n, p, P, ctrl_val=1) # If ctrl_val == 1: encodes to qr_z from qr_u2
print(ctrl_gate_O_X_c0.definition)
print(ctrl_gate_O_X_c1.definition)

                                                 
q_0: ─────o──────────o──────────o──────────o─────
          │          │          │          │     
q_1: ─────o──────────■──────────o──────────■─────
          │          │          │          │     
q_2: ─────o──────────o──────────■──────────■─────
     ┌────┴────┐┌────┴────┐┌────┴────┐┌────┴────┐
q_3: ┤0        ├┤0        ├┤0        ├┤0        ├
     │  X_u(3) ││  X_u(2) ││  X_u(1) ││  X_u(0) │
q_4: ┤1        ├┤1        ├┤1        ├┤1        ├
     └─────────┘└─────────┘└─────────┘└─────────┘
                                                 
q_0: ─────■──────────■──────────■──────────■─────
          │          │          │          │     
q_1: ─────o──────────■──────────o──────────■─────
          │          │          │          │     
q_2: ─────o──────────o──────────■──────────■─────
     ┌────┴────┐┌────┴────┐┌────┴────┐┌────┴────┐
q_3: ┤0        ├┤0        ├┤0        ├┤0        ├
     │  X_u(3) ││  X_u(2) ││  X_u(1) ││  X_u(0) │


In [133]:
########################################## Oracle O_K ###############################################

def oracle_O_K(nodes_degrees, n, k_bits):
    '''Return an oracle_O_K gate that encodes (k_v - 1) into qr_k, controlled on qr_v.'''
    qc = QuantumCircuit(n + k_bits, name='O_K')

    for v, k_v in enumerate(nodes_degrees):
        if k_v == 0:
            continue
        subgate = IntegerToRegisterGate('O_K_val', k_v - 1, k_bits)
        ctrl_gate = subgate.control(num_ctrl_qubits = n, ctrl_state = f'{v:0{n}b}')
        qc.append(ctrl_gate, list(range(n + k_bits)))

    return qc.to_gate(label='O_K')

ctrl_gate_O_K_c0 = oracle_O_K(nodes_degrees, n, 2*m).control(num_ctrl_qubits = 1, ctrl_state = f'{0:01b}')
ctrl_gate_O_K_c1 = oracle_O_K(nodes_degrees_c1, n, 2*m).control(num_ctrl_qubits = 1, ctrl_state = f'{1:01b}')
print(ctrl_gate_O_K_c0.definition)
print(ctrl_gate_O_K_c1.definition)

     ┌───┐        ┌───┐
q_0: ┤ X ├───■────┤ X ├
     └───┘┌──┴───┐└───┘
q_1: ─────┤0     ├─────
          │      │     
q_2: ─────┤1     ├─────
          │  O_K │     
q_3: ─────┤2     ├─────
          │      │     
q_4: ─────┤3     ├─────
          └──────┘     
                                                                               »
 control: ──■────■─────────■──────────■───■──────────■───■─────────■───■───────»
          ┌─┴─┐  │         │          │   │          │   │         │   │       »
target_0: ┤ X ├──┼─────────┼──────────┼───┼──────────■───┼─────────┼───┼───────»
          └───┘┌─┴─┐       │          │   │          │   │         │   │P(π/4) »
target_1: ─────┤ X ├───────┼──────────■───┼──────────┼───┼─────────■───■───────»
               └───┘┌──────┴───────┐┌─┴─┐ │P(-π/4) ┌─┴─┐ │P(π/4) ┌─┴─┐         »
target_2: ──────────┤ U(π/2,0,π,0) ├┤ X ├─■────────┤ X ├─■───────┤ X ├─────────»
                    └──────────────┘└───┘          └───┘         └───┘         »
target_

# QMME Circuit

## Initialization

In [134]:
################################### Quantum registers ######################################

qr_v = QuantumRegister(n, 'v')  # Basis-encode node index
qr_i = QuantumRegister(2, 'i')  # Basis-encode power (i) of feature value (x^i)
qr_c = QuantumRegister(1, 'c')  # Basis-encode power (c) of matrix A (A^c)
qr_l1 = QuantumRegister(m, 'l1')  # Basis-encode non-zero entries of each row of A
qr_l2 = QuantumRegister(m, 'l2')  # Basis-encode non-zero entries of each row of A^2
qr_u1 = QuantumRegister(n, 'u1')  # Basis-encode node's l-neighbor index of v (auxiliary)
qr_u2 = QuantumRegister(n, 'u2')  # Basis-encode node's l-neighbor index of u (auxiliary)
qr_k = QuantumRegister(2*m, 'k')  # Basis-encode degree value (k) [m for c=0, 2m for c=1]
qr_z = QuantumRegister(p, 'z')  # Basis-encode feature value (x)
qr_a = QuantumRegister(5, 'a')  # Ancilla qubit for rotation

''' Need n + 8 qubits always (i, c, a). At least, more 2n for u1/u2. Total of >= 3n + 8. '''

########### 0. Create the quantum circuit (Start with all qubits initialized to |0⟩) ########

qc = QuantumCircuit(qr_a, qr_z, qr_k, qr_u2, qr_u1, qr_l2, qr_l1, qr_c, qr_i, qr_v)

########## 1. Apply Hadamard gates to the `qr_v` qubits to create superposition #############

qc.h(qr_v)
print("\nExpected amplitudes:", 1/math.sqrt(N))


Expected amplitudes: 0.5


## Access Neighbors

In [135]:
############################ 2. Superposition of the l1-register qubits ######################

qc.h(qr_l1)
print("\nExpected amplitudes: ", 1/math.sqrt(N*s))

################################## 3. Apply Oracle O_L #######################################

qc.append(gate_O_L, [*qr_l1, *qr_v, *qr_u1])
print("\n adjacency_list: ", adjacency_list)

########################## 4. Superposition of the c-register qubits #########################

qc.h(qr_c) # For first and second-neighbors calculations


Expected amplitudes:  0.35355339059327373

 adjacency_list:  [[1, 3], [2, 0], [3, 1], [0, 2]]


In [136]:
#################################### CONTROLLED ON qr_c = 1 ############################################

################### 5. Superposition of the l2-register qubits controlled on qr_c ######################

qc.ch(control_qubit = qr_c, target_qubit = qr_l2) 
print(f"\nExpected amplitudes for qr_c being |0⟩ or |1⟩: {1/math.sqrt(2*N*s)} or {1/(s * math.sqrt(2*N))}")

############################### 6. Oracle O_L controlled on qr_c ########################################

ctrl_gate_O_L = oracle_O_L(adjacency_list, n, m).control(num_ctrl_qubits = 1)
qc.append(ctrl_gate_O_L, [qr_c, *qr_l2, *qr_u1, *qr_u2])

# access_statevector(qc) # 111m for (3,1,8)
print("\n adjacency_list: ", adjacency_list)


Expected amplitudes for qr_c being |0⟩ or |1⟩: 0.25 or 0.17677669529663687

 adjacency_list:  [[1, 3], [2, 0], [3, 1], [0, 2]]


## Feature Encoding

In [137]:
############################ 7. Superposition of the i-register qubits ######################

qc.h(qr_i)
print(f"\nExpected amplitudes for qr_c being |0⟩ or |1⟩: {1/(2 * math.sqrt(2*N*s))} or {1/(2 * s * math.sqrt(2*N))}")

######################################### 8. Oracle O_X ###############################################

qc.append(ctrl_gate_O_X_c0, [*qr_c, *qr_u1, *qr_z])  # qr_c = 0
qc.append(ctrl_gate_O_X_c1, [*qr_c, *qr_u2, *qr_z])  # qr_c = 1

# access_statevector(qc)
print("\nNode index and feature value: "), [print("node (u): ", u, features[u]) for u in range(N)]


Expected amplitudes for qr_c being |0⟩ or |1⟩: 0.125 or 0.08838834764831843

Node index and feature value: 
node (u):  0 0.75
node (u):  1 0.5
node (u):  2 0.25
node (u):  3 0.0


(None, [None, None, None, None])

### Controlled rotations

In [138]:
######################## 9. Controlled Rotation of Ancilla a1 by F_X ########################

thetas_F_X = [float(2 * np.arccos(x_int / P)) for x_int in range(P)]
F_X = FeatureRotation(thetas_F_X, P, "F_X")

def controlled_F_X(qc, qr_i, qr_a, qr_z, F_X):
    """ Apply the F_X transformation controlled by qr_i conditions.
        Does not need control on qr_c. Only matters qr_i and qr_z. """

    qc.append(F_X, [qr_a[1]] + qr_z[:])

    # Control qr_i conditions for qr_a targets
    control_map = {
        qr_a[2]: ["01", "10", "11"],
        qr_a[3]: ["10", "11"],
        qr_a[4]: ["11"]
    }

    for target, ctrl_states in control_map.items():
        for ctrl_state in ctrl_states:
            subgate_F_X = F_X.control(2, ctrl_state=ctrl_state)
            qc.append(subgate_F_X, [*qr_i, target, *qr_z])

    return qc

controlled_F_X(qc, qr_i, qr_a, qr_z, F_X)


Thetas (F_X):  [3.141592653589793, 2.636232143305636, 2.0943951023931953, 1.445468495626831]


In [139]:
### Access statevector (> 33 min for (3,1,8))

# access_statevector(qc)
print("\nExpected amplitudes (Node u):")
for u in range(N):
    for i in range(1, 5):
        for c in [0, 1]:
            if c == 0:
                amplitude = (features[u])**i / (2 * math.sqrt(2 * N * s))
            if c == 1:
                amplitude = (features[u])**i / (2 * s * math.sqrt(2 * N))
            print(f"Node u = {u}, x_u = {features[u]}, i = {i}, c = {c}, amplitude = {amplitude}")


Expected amplitudes (Node u):
Node u = 0, x_u = 0.75, i = 1, c = 0, amplitude = 0.09375
Node u = 0, x_u = 0.75, i = 1, c = 1, amplitude = 0.06629126073623882
Node u = 0, x_u = 0.75, i = 2, c = 0, amplitude = 0.0703125
Node u = 0, x_u = 0.75, i = 2, c = 1, amplitude = 0.04971844555217912
Node u = 0, x_u = 0.75, i = 3, c = 0, amplitude = 0.052734375
Node u = 0, x_u = 0.75, i = 3, c = 1, amplitude = 0.03728883416413434
Node u = 0, x_u = 0.75, i = 4, c = 0, amplitude = 0.03955078125
Node u = 0, x_u = 0.75, i = 4, c = 1, amplitude = 0.027966625623100753
Node u = 1, x_u = 0.5, i = 1, c = 0, amplitude = 0.0625
Node u = 1, x_u = 0.5, i = 1, c = 1, amplitude = 0.044194173824159216
Node u = 1, x_u = 0.5, i = 2, c = 0, amplitude = 0.03125
Node u = 1, x_u = 0.5, i = 2, c = 1, amplitude = 0.022097086912079608
Node u = 1, x_u = 0.5, i = 3, c = 0, amplitude = 0.015625
Node u = 1, x_u = 0.5, i = 3, c = 1, amplitude = 0.011048543456039804
Node u = 1, x_u = 0.5, i = 4, c = 0, amplitude = 0.0078125
Node

## Uncomputation

In [140]:
############################## 10. Oracle O_X_† (reset qr_z) ###############################

qc.append(ctrl_gate_O_X_c0, [*qr_c, *qr_u1, *qr_z])
qc.append(ctrl_gate_O_X_c1, [*qr_c, *qr_u2, *qr_z])

###################################### Second Neighbors ####################################

# 11. Oracle O_L_† (reset qr_u2) (controlled, again)
qc.append(ctrl_gate_O_L, [qr_c, *qr_l2, *qr_u1, *qr_u2])
# 12. Superposition in l2 (controlled, again)
qc.ch(control_qubit = qr_c, target_qubit = qr_l2) # x sqrt(s) where qr_c=1

###################################### First Neighbors ####################################

# 13. Oracle O_L_† (reset qr_u1)
qc.append(gate_O_L, [*qr_l1, *qr_v, *qr_u1])
# 14. Superposition in l1
qc.h(qr_l1) # x sqrt(s)

# access_statevector(qc, flag_l=False) # takes > 61 min for (3,1,8)

In [141]:
############################### Sum of neighbors' features' results #############################

from collections import defaultdict

expected_amplitudes = defaultdict(lambda: [[0, 0] for _ in range(4)])

for v in range(N):  
    for i in range(1, 5):  
        for u1 in adjacency_list[v]:  # First-level neighbors of v
            expected_amplitudes[v][i - 1][0] += (features[u1] ** i) / (2 * s * math.sqrt(2 * N))  # For u1
            for u2 in adjacency_list[u1]:  # Second-level neighbors of u1
                expected_amplitudes[v][i - 1][1] += (features[u2] ** i) / (2 * s**2 * math.sqrt(2 * N))  # For u2

print("\nExpected amplitudes:")
for v in sorted(expected_amplitudes):
    for i in range(4): 
        print(f"node v = {v}, i = {i}, c = 0, sum_features = {expected_amplitudes[v][i][0]}")
        print(f"node v = {v}, i = {i}, c = 1, sum_features = {expected_amplitudes[v][i][1]}")


Expected amplitudes:
node v = 0, i = 0, c = 0, sum_features = 0.044194173824159216
node v = 0, i = 0, c = 1, sum_features = 0.08838834764831843
node v = 0, i = 1, c = 0, sum_features = 0.022097086912079608
node v = 0, i = 1, c = 1, sum_features = 0.05524271728019903
node v = 0, i = 2, c = 0, sum_features = 0.011048543456039804
node v = 0, i = 2, c = 1, sum_features = 0.03866990209613931
node v = 0, i = 3, c = 0, sum_features = 0.005524271728019902
node v = 0, i = 3, c = 1, sum_features = 0.028311892606102
node v = 1, i = 0, c = 0, sum_features = 0.08838834764831843
node v = 1, i = 0, c = 1, sum_features = 0.044194173824159216
node v = 1, i = 1, c = 0, sum_features = 0.05524271728019903
node v = 1, i = 1, c = 1, sum_features = 0.022097086912079608
node v = 1, i = 2, c = 0, sum_features = 0.03866990209613931
node v = 1, i = 2, c = 1, sum_features = 0.011048543456039804
node v = 1, i = 3, c = 0, sum_features = 0.028311892606101997
node v = 1, i = 3, c = 1, sum_features = 0.00552427172801

## Division by degree

In [142]:
#################################### 15. Apply oracle O_K ####################################

qc.append(ctrl_gate_O_K_c0, [*qr_c, *qr_v, *qr_k])  # qr_c = 0
qc.append(ctrl_gate_O_K_c1, [*qr_c, *qr_v, *qr_k])  # qr_c = 1

In [143]:
######################## 16. Controlled Rotation of Ancilla a1 by F_X ######################## (> 1 min for (2,1,4))

thetas_F_K_inv = [float(2 * np.arccos(1 / (k_int + 1))) for k_int in range(2*s)] # [Times two comes from using 2m qubits]
F_K_inv = FeatureRotation(thetas_F_K_inv, 2*s, "F_K_inv")

qc.append(F_K_inv, [qr_a[0]] + qr_k[:])
# access_statevector(qc, flag_a=False) #, flag_l=False)
print("\nExpected amplitudes:")
for v in sorted(expected_amplitudes):
    for i in range(4): 
        print(f"node v = {v}, i = {i}, c = 0, sum_features = {expected_amplitudes[v][i][0]/nodes_degrees[v]}")
        print(f"node v = {v}, i = {i}, c = 1, sum_features = {expected_amplitudes[v][i][1]/nodes_degrees_c1[v]}")

# Uncompute (reset qr_k)
qc.append(ctrl_gate_O_K_c0, [*qr_c, *qr_v, *qr_k])  # qr_c = 0
qc.append(ctrl_gate_O_K_c1, [*qr_c, *qr_v, *qr_k])  # qr_c = 1


Thetas (F_X):  [0.0, 2.0943951023931953, 2.4619188346815495, 2.636232143305636]

Expected amplitudes:
node v = 0, i = 0, c = 0, sum_features = 0.022097086912079608
node v = 0, i = 0, c = 1, sum_features = 0.022097086912079608
node v = 0, i = 1, c = 0, sum_features = 0.011048543456039804
node v = 0, i = 1, c = 1, sum_features = 0.013810679320049757
node v = 0, i = 2, c = 0, sum_features = 0.005524271728019902
node v = 0, i = 2, c = 1, sum_features = 0.009667475524034828
node v = 0, i = 3, c = 0, sum_features = 0.002762135864009951
node v = 0, i = 3, c = 1, sum_features = 0.0070779731515255
node v = 1, i = 0, c = 0, sum_features = 0.044194173824159216
node v = 1, i = 0, c = 1, sum_features = 0.011048543456039804
node v = 1, i = 1, c = 0, sum_features = 0.027621358640099514
node v = 1, i = 1, c = 1, sum_features = 0.005524271728019902
node v = 1, i = 2, c = 0, sum_features = 0.019334951048069655
node v = 1, i = 2, c = 1, sum_features = 0.002762135864009951
node v = 1, i = 3, c = 0, sum_f

# Draw and Visualize

In [ ]:
###################################### Draw and Visualize ########################################

print("\n -------------------------------------------------------------------------------\n")
# circuit_drawer(qc, output='mpl', filename='network_circuit_qmme.png')
# qc.draw(output='mpl', filename='network_circuit_qmme.png')


 -------------------------------------------------------------------------------



MissingOptionalLibraryError: "The 'pylatexenc' library is required to use 'MatplotlibDrawer'. You can install it with 'pip install pylatexenc'."

In [ ]:
with open("QMME_circuit_4_nodes.qpy", "wb") as f:
    qpy.dump(qc, f)